Change the corresponding parameters in the config json files

In [1]:
import json
import os

root_folder = "/home/yanhongwei/DGIL/DGIL/configs/DGIL_baseline_3_run"

for root, dirs, files in os.walk(root_folder):
    for file in files:
        if file.endswith(".json"):
            with open(os.path.join(root, file), "r") as f:
                config = json.load(f)
            config["seed"] = [1994, 1995, 1996]
            with open(os.path.join(root, file), "w") as f:
                json.dump(config, f, indent=4)

Check whether the config files are valid or not

In [3]:
import json
import os

# root_folder = "/home/yanhongwei/DGIL/DGIL/configs/DGIL_baseline_3_run"
# root_folder = "/home/yanhongwei/DGIL/DGIL/configs/DGIL"
root_folder = "/gpfs-flash/junlab/liyuan/hongwei/DGIL/configs/DGIL"

dataset_cls_map = {
    "core50": 5,
    "imageclef": 2,
    "office31": 4,
    "officecaltech": 2,
    "officehome": 13,
    "digitsdg": 2,
    "digitsfive": 2,
    "core50": 5,
    "minidomainnet": 13,
    "domainnet": 35
}

total_json_files = 0
valid_data_json_files = 0
cil_json_files = 0
dgil_json_files = 0

for root, dirs, files in os.walk(root_folder):
    for file in files:
        if file.endswith(".json"):
            total_json_files += 1
            with open(os.path.join(root, file), "r") as f:
                config = json.load(f)

            data_checked = False
            for key in dataset_cls_map:
                if config["dataset"] == key and key in root:
                    valid_data_json_files += 1
                    assert config["init_cls"] == config["increment"] == dataset_cls_map[key], f"file: {root} {file}, dataset: {config['dataset']}, init_cls: {config['init_cls']}, increment: {config['increment']}, expected: {dataset_cls_map[key]}"
                    data_checked = True
                    break
            assert data_checked, f"file: {root} {file}, dataset: {config['dataset']}, init_cls: {config['init_cls']}, increment: {config['increment']}"


            dgil_checked = False
            if "dgil" in file:
                dgil_json_files += 1
                assert config["enable_dgil"] == True, f"file: {root} {file}, enable_dgil: {config['enable_dgil']}"
                assert config["random_reference"] == True, f"file: {root} {file}, random_reference: {config['random_reference']}"
                assert config["reference_domain_id"] == 0, f"file: {root} {file}, reference_domain_id: {config['reference_domain_id']}"
                assert config["multi_domain_base_task"] == False, f"file: {root} {file}, multi_domain_base_task: {config['multi_domain_base_task']}"
            else:
                cil_json_files += 1
                dgil_checked = True

            assert config["print_forget"] == True


print(f"total_json_files: {total_json_files}, valid_data_json_files: {valid_data_json_files}, cil_json_files: {cil_json_files}, dgil_json_files: {dgil_json_files}")

total_json_files: 243, valid_data_json_files: 243, cil_json_files: 63, dgil_json_files: 180


Generate configs based on template config

In [ ]:
import json
import os
from pprint import pprint

root_folder = "/gpfs-flash/junlab/liyuan/hongwei/DGIL/configs/DGIL/officehome"
# template_cil_file = os.path.join(root_folder, "dot.json")
template_dgil_file = os.path.join(root_folder, "slca_dgil.json")

# with open(template_cil_file, "r") as f:
#     cil_config = json.load(f)

with open(template_dgil_file, "r") as f:
    dgil_config = json.load(f)

for weight in [0.01, 0.1, 1, 10]:
    post_fix = f"mmda_{weight:.2f}"
    # new_cil_file = os.path.join(root_folder, f"dot_{post_fix}.json")
    new_dgil_file = os.path.join(root_folder, f"slca_dgil_{post_fix}.json")

    # print(f"Generating new_cil_file: {new_cil_file}")
    print(f"Generating new_dgil_file: {new_dgil_file}")

    # new_cil_config = cil_config.copy()
    # new_cil_config["prefix"] = f"CIL_{post_fix}"
    # new_cil_config["use_dom_con_loss"] = True
    # new_cil_config["dom_con_weight"] = weight

    new_dgil_config = dgil_config.copy()
    new_dgil_config["prefix"] = f"DGIL_{post_fix}"
    new_dgil_config["mmda_loss_weight"] = weight

    # with open(new_cil_file, "w") as f:
    #     json.dump(new_cil_config, f, indent=4)

    with open(new_dgil_file, "w") as f:
        json.dump(new_dgil_config, f, indent=4)

Generating new_dgil_file: /gpfs-flash/junlab/liyuan/hongwei/DGIL/configs/DGIL/officehome/slca_dgil_mmda_0.01.json
Generating new_dgil_file: /gpfs-flash/junlab/liyuan/hongwei/DGIL/configs/DGIL/officehome/slca_dgil_mmda_0.10.json
Generating new_dgil_file: /gpfs-flash/junlab/liyuan/hongwei/DGIL/configs/DGIL/officehome/slca_dgil_mmda_1.00.json
Generating new_dgil_file: /gpfs-flash/junlab/liyuan/hongwei/DGIL/configs/DGIL/officehome/slca_dgil_mmda_10.00.json


Generate variantions of config files for different experiments.

In [6]:
import json
import os
from pprint import pprint

root_folder = "/gpfs-flash/junlab/liyuan/hongwei/DGIL/configs/DGIL/officehome"
output_folder = f"{root_folder}/dot_slca_hpp"
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

example_name = "dot_slca_dgil_randaug"

example_config_file = os.path.join(root_folder, f"{example_name}.json")
with open(example_config_file, "r") as f:
    example_config = json.load(f)

for lbd in [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]:
    if lbd == 1:
        dom_loss_weight = 5
    else:
        dom_loss_weight = lbd / (1-lbd)
    print(f"dom_loss_weight: {dom_loss_weight}")
    post_fix = f"_lbd{lbd}"
    new_config_file = os.path.join(output_folder, f"{example_name}{post_fix}.json")
    new_config = example_config.copy()
    example_prefix = new_config["prefix"]
    new_config["prefix"] = f"{example_prefix}{post_fix}"
    new_config["dom_loss_weight"] = dom_loss_weight
    new_config["seed"] = [1994, 1995, 1996]
    with open(new_config_file, "w") as f:
        json.dump(new_config, f, indent=4)


for dot_epochs in [2, 4, 6, 8, 10, 12]:
    post_fix = f"_de{dot_epochs}"
    new_config_file = os.path.join(output_folder, f"{example_name}{post_fix}.json")
    new_config = example_config.copy()
    example_prefix = new_config["prefix"]
    new_config["prefix"] = f"{example_prefix}{post_fix}"
    new_config["dot_epochs"] = dot_epochs
    new_config["seed"] = [1994, 1995, 1996]
    with open(new_config_file, "w") as f:
        json.dump(new_config, f, indent=4)


for domain_centroids in [1, 2, 4, 8, 16, 32, 64]:
    post_fix = f"_dc{domain_centroids}"
    new_config_file = os.path.join(output_folder, f"{example_name}{post_fix}.json")
    new_config = example_config.copy()
    example_prefix = new_config["prefix"]
    new_config["prefix"] = f"{example_prefix}{post_fix}"
    new_config["domain_centroids"] = domain_centroids
    new_config["seed"] = [1994, 1995, 1996]
    with open(new_config_file, "w") as f:
        json.dump(new_config, f, indent=4)


dom_loss_weight: 0.0
dom_loss_weight: 0.25
dom_loss_weight: 0.6666666666666667
dom_loss_weight: 1.4999999999999998
dom_loss_weight: 4.000000000000001
dom_loss_weight: 5
